# Step 2: Data Cleaning, Wrangling & Pre-Processing #

## Goals: ##
* Clean and wrangle dataset. This could include dropping null values, removing unnecessary columns, removing outliers, and potentially fixing incorrectly formatted data.
* Save this dataframe as a new csv file to be used in the next step.


## Import packages ##

In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np 

df = pd.read_csv("../data/bank_transactions.csv")

## Drop null values ##

EDA has shown there are no null values.

## Drop unncessary columns ##

I'm dropping `nameOrig` and `nameDest` because they are identifier-like fields rather than behavioral predictors. They could potentially help identify specific accounts, but they do not directly describe the transaction patterns that drive fraud, and keeping them would risk pushing the model toward memorizing account IDs instead of learning generalizable fraud behavior.

I also dropped `isFlaggedFraud` because it is an extremely sparse pre-existing flag and is not a reliable feature for modeling fraud behavior in this dataset. In this data, it is only positive for a tiny number of rows and appears to be inconsistent with the fraud label, so it is better treated as a descriptive artifact than as a useful predictor.


In [9]:
df = df.drop(columns=["nameOrig", "nameDest", "isFlaggedFraud"])

## Feature Engineering ##

I'm creating time- and balance-based features because the EDA suggests fraud is linked to late-night activity and to specific balance changes around the transaction. The hour feature captures time-of-day effects, while the balance-difference and near-equality flags encode the fraud signature more directly, which should help tree-based models learn rule-like patterns more effectively than relying on raw variables alone.


In [10]:
df["hour"] = df["step"] % 24
df["diff_orig"] = df["oldbalanceOrg"] - df["newbalanceOrig"]
df["diff_dest"] = df["newbalanceDest"] - df["oldbalanceDest"]

df["amt_equals_orig_balance"] = (np.isclose(df["amount"], df["oldbalanceOrg"], rtol=0, atol=1)).astype(int)
df["orig_balance_zero_after"] = (df["newbalanceOrig"] == 0).astype(int)
df["dest_balance_unchanged"] = (np.isclose(df["newbalanceDest"], df["oldbalanceDest"], rtol=0, atol=1)).astype(int)


Because `DEBIT` and `PAYMENT` contain no fraud cases in this dataset, I plan to compare two modeling approaches: one trained on all transaction types and one trained only on `TRANSFER` and `CASH_OUT`. This makes it possible to test whether removing the zero-fraud types improves fraud detection without sacrificing performance on unseen data.


In [11]:
df_all_types = df
df_only_fraud_types = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])]

`df_all_types` needs dummy encoding for the four types but `df_only_fraud_types` wouldn't need it since it would have only 2 types.

In [12]:
df_all_types = pd.get_dummies(df, columns=['type'], drop_first=True)
df_all_types.head()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,hour,diff_orig,diff_dest,amt_equals_orig_balance,orig_balance_zero_after,dest_balance_unchanged,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER
0,1,9839.64,170136.0,160296.36,0.0,0.0,0,1,9839.64,0.0,0,0,1,False,False,True,False
1,1,1864.28,21249.0,19384.72,0.0,0.0,0,1,1864.28,0.0,0,0,1,False,False,True,False
2,1,181.00,181.0,0.00,0.0,0.0,1,1,181.00,0.0,1,1,1,False,False,False,True
3,1,181.00,181.0,0.00,21182.0,0.0,1,1,181.00,-21182.0,1,1,0,True,False,False,False
4,1,11668.14,41554.0,29885.86,0.0,0.0,0,1,11668.14,0.0,0,0,1,False,False,True,False


In [15]:
df_only_fraud_types.head()

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,hour,diff_orig,diff_dest,amt_equals_orig_balance,orig_balance_zero_after,dest_balance_unchanged
2,1,TRANSFER,181.00,181.0,0.0,0.0,0.00,1,1,181.0,0.00,1,1,1
3,1,CASH_OUT,181.00,181.0,0.0,21182.0,0.00,1,1,181.0,-21182.00,1,1,0
15,1,CASH_OUT,229133.94,15325.0,0.0,5083.0,51513.44,0,1,15325.0,46430.44,0,1,0
19,1,TRANSFER,215310.30,705.0,0.0,22425.0,0.00,0,1,705.0,-22425.00,0,1,0
24,1,TRANSFER,311685.89,10835.0,0.0,6267.0,2719172.89,0,1,10835.0,2712905.89,0,1,0


## Remove outliers ##


I've decided in EDA not to drop the outeliers due to the extreme right skew of most of the data

## Save cleaned data ##

In [14]:
df_all_types.to_csv("../data/cleaned_all_types_bank_transactions.csv", index=False)
df_only_fraud_types.to_csv("../data/cleaned_only_fraud_types_bank_transactions.csv", index=False)